In [2]:
import pandas as pd
import numpy as np
import torch
import os
import huggingface_hub
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from tqdm import tqdm
import ast
import math
import argparse
from sklearn.preprocessing import StandardScaler
import re
from pathlib import Path
import os
import spacy

# procedure:
1. gather all the data in long format (results for each model concatenated below each other)
    1.a. calculate frequency as the sum of the frames from the sketch engine
2. center the data
3. main predictors: predictability, frequency
    random intercept: item, model
    dependent variable: decomposability

In [3]:
os.chdir("/Users/golzaratefi/Documents/idioms_decomposability")
print(os.getcwd())


/Users/golzaratefi/Documents/idioms_decomposability


In [4]:
impli_raw = "data/impli/checked_manual_e.csv"
frequency = "data/frequencies_infini/impli/frequencies_infini_impli_v4_olmo-2-1124-13b-instruct_llama.csv"
decomp = ["decomp_measure/scores/impli_layers/google-bert_bert-large-uncased/2025-12-25_02:34:13/layer_23/impli_wasser_sum_bert-large-uncased.csv",
          "decomp_measure/scores/impli_layers/answerdotai_ModernBERT-base/2025-12-25_02:34:31/layer_9/impli_cka_sum_ModernBERT-base.csv",
          "decomp_measure/scores/impli_layers/google-bert_bert-large-cased/2025-12-25_02:34:13/layer_23/impli_cka_gini_bert-large-cased.csv",
          "decomp_measure/scores/impli_layers/answerdotai_ModernBERT-large/2025-12-25_02:34:31/layer_17/impli_wasser_sum_ModernBERT-large.csv",
          "decomp_measure/scores/impli_layers/google-bert_bert-base-cased/2025-12-25_02:34:13/layer_1/impli_wasser_mean_bert-base-cased.csv",
          "decomp_measure/scores/impli_layers/google-bert_bert-base-uncased/2025-12-25_02:34:13/layer_1/impli_wasser_mean_bert-base-uncased.csv"
          ""
        
]
predictability = "predictability/results/pred"

# getting decomp

In [5]:
total_decomp = pd.DataFrame()

for file in decomp:
    
    decomp_file = pd.read_csv(file)
    # decomp_data = decomp_file["decomp_score"]
    p = Path(file)
    model = p.parts[3] 
    cols = ["premise", "hypothesis", "extracted_idiom"]
    decomp_file = decomp_file.drop(decomp_file.columns[[0,5,6,7,9]], axis=1)
    decomp_file["model"] = model
    total_decomp = pd.concat([total_decomp, decomp_file], axis=0)

In [6]:
total_decomp

,premise,hypothesis,extracted_idiom,base_form,decomp_score,model
0,How have you weathered the storm ?,How have you succeeded in getting through the ...,weathered the storm,weather the storm,0.017583,google-bert_bert-large-uncased
1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,0.101737,google-bert_bert-large-uncased
2,He prefers acting with other countries to goin...,He prefers acting with other countries to doin...,going it alone,go it alone,0.039772,google-bert_bert-large-uncased
3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,0.110996,google-bert_bert-large-uncased
4,"Attractive , popular and smart , Veronica hove...","Attractive , popular and smart , Veronica hove...",rule the roost,rule the roost,0.165083,google-bert_bert-large-uncased
...,...,...,...,...,...,...
522,"Her arm linked through his , publicly staking ...","Her arm linked through his , publicly making a...",staking her claim,stake a claim,0.057166,google-bert_bert-base-uncased
523,The various departmental select committees loo...,The various departmental select committees loo...,scratch the surface,scratch the surface,0.030369,google-bert_bert-base-uncased
524,GRAHAM TAYLOR searches desperately for someone...,GRAHAM TAYLOR searches desperately for someone...,get a grip,get a grip,0.033121,google-bert_bert-base-uncased
525,Villa Sylvia — free and easy holiday style,Villa Sylvia — relaxed holiday style,free and easy,free and easy,0.015972,google-bert_bert-base-uncased


In [51]:
total_decomp.groupby("model").get_group("answerdotai_ModernBERT-base")


,premise,hypothesis,extracted_idiom,base_form,decomp_score,model
0,How have you weathered the storm ?,How have you succeeded in getting through the ...,weathered the storm,weather the storm,0.192430,answerdotai_ModernBERT-base
1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,0.255896,answerdotai_ModernBERT-base
2,He prefers acting with other countries to goin...,He prefers acting with other countries to doin...,going it alone,go it alone,0.507726,answerdotai_ModernBERT-base
3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,0.078304,answerdotai_ModernBERT-base
4,"Attractive , popular and smart , Veronica hove...","Attractive , popular and smart , Veronica hove...",rule the roost,rule the roost,3.726450,answerdotai_ModernBERT-base
...,...,...,...,...,...,...
522,"Her arm linked through his , publicly staking ...","Her arm linked through his , publicly making a...",staking her claim,stake a claim,0.178539,answerdotai_ModernBERT-base
523,The various departmental select committees loo...,The various departmental select committees loo...,scratch the surface,scratch the surface,0.058798,answerdotai_ModernBERT-base
524,GRAHAM TAYLOR searches desperately for someone...,GRAHAM TAYLOR searches desperately for someone...,get a grip,get a grip,0.154088,answerdotai_ModernBERT-base
525,Villa Sylvia — free and easy holiday style,Villa Sylvia — relaxed holiday style,free and easy,free and easy,0.226828,answerdotai_ModernBERT-base


# getting predictability

In [8]:
total_pred = pd.DataFrame()
pred = os.listdir(predictability)
for file in pred:
    file = os.path.join(predictability, file)
    pred_file = pd.read_csv(file)
    # decomp_data = decomp_file["decomp_score"]
    p = Path(file)
    model = p.parts[3] 
    if "ModernBERT-base" in model:
        model = "answerdotai_ModernBERT-base"
    elif "ModernBERT-large" in model:
        model = "answerdotai_ModernBERT-large"
    elif "bert-large-uncased" in model:
        model = "google-bert_bert-large-uncased"
    elif "bert-large-cased" in model:
        model = "google-bert_bert-large-cased"
    elif "bert-base-cased" in model:
        model = "google-bert_bert-base-cased"
    elif "bert-base-uncased" in model:
        model = "google-bert_bert-base-uncased"
    else:
        raise ValueError("Unknown model name in file path.")
    pred_file["model"] = model
    total_pred = pd.concat([total_pred, pred_file], axis=0)

In [50]:
total_pred.groupby("model").get_group("answerdotai_ModernBERT-base")

,Unnamed: 0,premise,hypothesis,extracted_idiom,base_form,input,target,prob,log_prob,norm_log_prob,model
0,0,How have you weathered the storm ?,How have you succeeded in getting through the ...,weathered the storm,weather the storm,How have you weathered the storm ?,storm,0.589844,-0.527344,-0.527344,answerdotai_ModernBERT-base
1,1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,"And then , the Daily Telegraph discovered ‘ th...",grain,0.871094,-0.137695,-0.137695,answerdotai_ModernBERT-base
2,2,He prefers acting with other countries to goin...,He prefers acting with other countries to doin...,going it alone,go it alone,He prefers acting with other countries to goin...,alone,0.996094,-0.003906,-0.003906,answerdotai_ModernBERT-base
3,3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,The imagination trembles at some of these idea...,clean,0.687500,-0.375000,-0.375000,answerdotai_ModernBERT-base
4,4,"Attractive , popular and smart , Veronica hove...","Attractive , popular and smart , Veronica hove...",rule the roost,rule the roost,"Attractive , popular and smart , Veronica hove...",roost,0.996094,-0.003906,-0.003906,answerdotai_ModernBERT-base
...,...,...,...,...,...,...,...,...,...,...,...
522,522,"Her arm linked through his , publicly staking ...","Her arm linked through his , publicly making a...",staking her claim,stake a claim,"Her arm linked through his , publicly staking ...",claim,0.066406,-2.718750,-2.718750,answerdotai_ModernBERT-base
523,523,The various departmental select committees loo...,The various departmental select committees loo...,scratch the surface,scratch the surface,The various departmental select committees loo...,surface,0.992188,-0.007874,-0.007874,answerdotai_ModernBERT-base
524,524,GRAHAM TAYLOR searches desperately for someone...,GRAHAM TAYLOR searches desperately for someone...,get a grip,get a grip,GRAHAM TAYLOR searches desperately for someone...,grip,0.059570,-2.828125,-2.828125,answerdotai_ModernBERT-base
525,525,Villa Sylvia — free and easy holiday style,Villa Sylvia — relaxed holiday style,free and easy,free and easy,Villa Sylvia — free and easy holiday style,easy,0.029907,-3.515625,-3.515625,answerdotai_ModernBERT-base


# getting frequency

In [57]:
frequency_file = pd.read_csv("data/frequencies/impli/total_frequencies.csv")

In [58]:
# replace NaN values with 0
frequency_file['adj_insertion_full_1'] = frequency_file['adj_insertion_full_1'].fillna(0)
frequency_file['adj_insertion_full_2'] = frequency_file['adj_insertion_full_2'].fillna(0)
frequency_file['adv_insertion_full'] = frequency_file['adv_insertion_full'].fillna(0)
frequency_file['adv_insertion_full_adj'] = frequency_file['adv_insertion_full_adj'].fillna(0)
frequency_file['identity_full'] = frequency_file['identity_full'].fillna(0)
frequency_file['nominalization_full'] = frequency_file['nominalization_full'].fillna(0)
frequency_file['passive_full'] = frequency_file['passive_full'].fillna(0)

In [59]:
frequency_file["nominalization_full"][3]

0.0

In [60]:
total_frequency = frequency_file["adj_insertion_full_1"] + frequency_file["adj_insertion_full_2"] + frequency_file["adv_insertion_full"] + frequency_file["adv_insertion_full_adj"] + frequency_file["identity_full"] + frequency_file["nominalization_full"] + frequency_file["passive_full"]

In [61]:
total_frequency

0       20020.0
1       29508.0
2       34464.0
3       55128.0
4       12572.0
         ...   
522     10759.0
523     80926.0
524     43117.0
525     15627.0
526    236908.0
Length: 527, dtype: float64

# getting idiom structures

In [20]:
nlp = spacy.load("en_core_web_sm")


In [21]:
def coarse_shape_from_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return "EMPTY"
    doc = nlp(text)
    return coarse_shape(doc)

POS_KEEP = {"NOUN","PROPN","PRON","VERB","AUX","ADJ","ADV","ADP","DET","PART","CCONJ","SCONJ","NUM","INTJ"}
def pos_signature(doc):
    sig = []
    for t in doc:
        if t.is_punct or t.is_space:
            continue
        p = t.pos_
        sig.append(p if p in POS_KEEP else "X")
    return " ".join(sig)

def coarse_shape(doc):

    sent = next(doc.sents)
    root = sent.root

    # Clause-like
    if root.pos_ in {"VERB", "AUX"}:
        # If it looks like an imperative or has a subject, treat as VP/clause
        has_subj = any(ch.dep_ in {"nsubj", "nsubjpass", "csubj", "expl"} for ch in root.children)
        return "S" if has_subj else "VP"

    # Noun phrase
    if root.pos_ in {"NOUN", "PROPN", "PRON"}:
        return "NP"

    # Prepositional phrase often has ADP as root in fragments ("in a nutshell")
    if root.pos_ == "ADP":
        return "PP"

    # Adjective/adverb phrases
    if root.pos_ == "ADJ":
        return "ADJP"
    if root.pos_ == "ADV":
        return "ADVP"

    return f"OTHER({root.pos_})"

In [22]:
raw_data = pd.read_csv(impli_raw)

In [23]:
structure = raw_data["base_form"].apply(coarse_shape_from_text)

In [24]:
structure

0        VP
1        PP
2        VP
3        VP
4        VP
       ... 
522      VP
523      VP
524      VP
525    ADJP
526      PP
Name: base_form, Length: 527, dtype: object

# gathering all_data

In [43]:
Total_data = total_decomp.merge(total_pred, how="left", on=["premise", "hypothesis", "extracted_idiom", "base_form", "model"])

In [44]:
Total_data

,premise,hypothesis,extracted_idiom,base_form,decomp_score,model,Unnamed: 0,input,target,prob,log_prob,norm_log_prob
0,How have you weathered the storm ?,How have you succeeded in getting through the ...,weathered the storm,weather the storm,0.017583,google-bert_bert-large-uncased,0.0,How have you weathered the storm ?,storm,0.357422,-1.031250,-1.031250
1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,0.101737,google-bert_bert-large-uncased,1.0,"And then , the Daily Telegraph discovered ‘ th...",grain,0.292969,-1.226562,-1.226562
2,He prefers acting with other countries to goin...,He prefers acting with other countries to doin...,going it alone,go it alone,0.039772,google-bert_bert-large-uncased,2.0,He prefers acting with other countries to goin...,alone,0.964844,-0.035889,-0.035889
3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,0.110996,google-bert_bert-large-uncased,3.0,The imagination trembles at some of these idea...,clean,0.253906,-1.367188,-1.367188
4,"Attractive , popular and smart , Veronica hove...","Attractive , popular and smart , Veronica hove...",rule the roost,rule the roost,0.165083,google-bert_bert-large-uncased,4.0,"Attractive , popular and smart , Veronica hove...",roost,0.000016,-11.000000,-5.500000
...,...,...,...,...,...,...,...,...,...,...,...,...
3157,"Her arm linked through his , publicly staking ...","Her arm linked through his , publicly making a...",staking her claim,stake a claim,0.057166,google-bert_bert-base-uncased,522.0,"Her arm linked through his , publicly staking ...",claim,0.968750,-0.031738,-0.031738
3158,The various departmental select committees loo...,The various departmental select committees loo...,scratch the surface,scratch the surface,0.030369,google-bert_bert-base-uncased,523.0,The various departmental select committees loo...,surface,0.519531,-0.656250,-0.656250
3159,GRAHAM TAYLOR searches desperately for someone...,GRAHAM TAYLOR searches desperately for someone...,get a grip,get a grip,0.033121,google-bert_bert-base-uncased,524.0,GRAHAM TAYLOR searches desperately for someone...,grip,0.002670,-5.937500,-5.937500
3160,Villa Sylvia — free and easy holiday style,Villa Sylvia — relaxed holiday style,free and easy,free and easy,0.015972,google-bert_bert-base-uncased,525.0,Villa Sylvia — free and easy holiday style,easy,0.000931,-6.968750,-6.968750


In [65]:
total_frequency_expand = pd.concat([total_frequency] * 6, ignore_index=True)

In [68]:
structure_expand = pd.concat([structure] * 6, ignore_index=True)

In [72]:
Total_data["structure"] = structure_expand
Total_data["frequency"] = total_frequency_expand

In [73]:
Total_data = Total_data.rename(columns={"norm_log_prob": "predictability_score"})


In [74]:
Total_data

,premise,hypothesis,extracted_idiom,base_form,decomp_score,model,Unnamed: 0,input,target,prob,log_prob,predictability_score,structure,frequency
0,How have you weathered the storm ?,How have you succeeded in getting through the ...,weathered the storm,weather the storm,0.017583,google-bert_bert-large-uncased,0.0,How have you weathered the storm ?,storm,0.357422,-1.031250,-1.031250,VP,20020.0
1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,0.101737,google-bert_bert-large-uncased,1.0,"And then , the Daily Telegraph discovered ‘ th...",grain,0.292969,-1.226562,-1.226562,PP,29508.0
2,He prefers acting with other countries to goin...,He prefers acting with other countries to doin...,going it alone,go it alone,0.039772,google-bert_bert-large-uncased,2.0,He prefers acting with other countries to goin...,alone,0.964844,-0.035889,-0.035889,VP,34464.0
3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,0.110996,google-bert_bert-large-uncased,3.0,The imagination trembles at some of these idea...,clean,0.253906,-1.367188,-1.367188,VP,55128.0
4,"Attractive , popular and smart , Veronica hove...","Attractive , popular and smart , Veronica hove...",rule the roost,rule the roost,0.165083,google-bert_bert-large-uncased,4.0,"Attractive , popular and smart , Veronica hove...",roost,0.000016,-11.000000,-5.500000,VP,12572.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3157,"Her arm linked through his , publicly staking ...","Her arm linked through his , publicly making a...",staking her claim,stake a claim,0.057166,google-bert_bert-base-uncased,522.0,"Her arm linked through his , publicly staking ...",claim,0.968750,-0.031738,-0.031738,VP,10759.0
3158,The various departmental select committees loo...,The various departmental select committees loo...,scratch the surface,scratch the surface,0.030369,google-bert_bert-base-uncased,523.0,The various departmental select committees loo...,surface,0.519531,-0.656250,-0.656250,VP,80926.0
3159,GRAHAM TAYLOR searches desperately for someone...,GRAHAM TAYLOR searches desperately for someone...,get a grip,get a grip,0.033121,google-bert_bert-base-uncased,524.0,GRAHAM TAYLOR searches desperately for someone...,grip,0.002670,-5.937500,-5.937500,VP,43117.0
3160,Villa Sylvia — free and easy holiday style,Villa Sylvia — relaxed holiday style,free and easy,free and easy,0.015972,google-bert_bert-base-uncased,525.0,Villa Sylvia — free and easy holiday style,easy,0.000931,-6.968750,-6.968750,ADJP,15627.0


In [75]:
Total_data = Total_data[["premise", "hypothesis", "extracted_idiom", "base_form", "model", "decomp_score", "predictability_score", "structure", "frequency"]]

In [77]:
os.makedirs("mixed_effect_analysis", exist_ok=True)
Total_data.to_csv("mixed_effect_analysis/impli_mixed_effect_data.csv", index=False)

# standardize data

In [ ]:


scaler = StandardScaler()
dataset[["predictability", "frequency", "surprisal"]] = scaler.fit_transform(
    dataset[["predictability", "frequency", "surprisal"]]
)